In [0]:
%sql DESCRIBE EXTERNAL LOCATION ucext;

In [0]:
%sql
--  DROP SCHEMA IF EXISTS man_catlog.caresync_schema_bronze CASCADE;
-- DROP SCHEMA IF EXISTS man_catlog.caresync_schema_gold CASCADE;
-- DROP SCHEMA IF EXISTS man_catlog.caresync_schema_landing CASCADE;
-- DROP SCHEMA IF EXISTS man_catlog.caresync_schema_silver CASCADE;
-- DROP SCHEMA IF EXISTS man_catlog.caresync_schema_bronze CASCADE;


In [0]:
dbutils.fs.ls("/Volumes/man_catlog/landing/landing_volume/neon_postgres/doctors/2026/11/24")

In [0]:
# dbutils.fs.ls("/Volumes/man_catlog/landing/landing_volume/neon_postgres/hospitals/2026/11/23/")

In [0]:

dbutils.widgets.text('table_name','')
dbutils.widgets.text('landing_timestamp','')
dbutils.widgets.text('merge_keys','')
dbutils.widgets.text('load_type','')
# dbutils.widgets.text('year','')
# dbutils.widgets.text('month','')
# dbutils.widgets.text('date','')


table_name = dbutils.widgets.get('table_name')
landing_timestamp = dbutils.widgets.get('landing_timestamp')
merge_keys = dbutils.widgets.get('merge_keys')
load_type = dbutils.widgets.get('load_type')
# landing_timestamp = dbutils.widgets.get('year') + '/' + dbutils.widgets.get('month') + '/' + dbutils.widgets.get('date')+'/'
# file_path = '/Volumes/man_catlog/landing/landing_volume/' + source_system + '/' + table_name + '/' + landing_timestamp

In [0]:
bronze_df = spark.read.table(f"man_catlog.caresync_schema_bronze.{table_name}").filter(f"landing_timestamp = '{landing_timestamp}'")

In [0]:
from pyspark.sql import functions as f
bronze_updt_df = bronze_df.withColumn('current_ts',f.current_timestamp())\
                     .withColumn('landing_ts',f.current_timestamp())
bronze_updt_df.show(5)

In [0]:
from delta.tables import DeltaTable
from pyspark.sql import Window

target_table = f"man_catlog.caresync_schema_silver.{table_name}"
load_type_norm = load_type.strip().upper()

if load_type_norm in ("FULL", "OVERWRITE", ""):
    # Full load: replace target data
    bronze_updt_df.write.mode("overwrite").option("overwriteSchema", "true") \
                  .saveAsTable(target_table)

elif load_type_norm == "APPEND":
    # Append: add incoming rows to target as-is
    bronze_updt_df.write.mode("append").saveAsTable(target_table)

elif load_type_norm == "MERGE":
    # Merge (upsert): update existing rows, insert new ones
    keys = [k.strip() for k in merge_keys.split(",") if k.strip()]
    if not keys:
        raise ValueError("merge_key widget is required for MERGE load (comma-separated for composite keys)")

    if not spark.catalog.tableExists(target_table):
        # First run: target doesn't exist yet, create it
        bronze_updt_df.write.mode("overwrite").saveAsTable(target_table)
    else:
        # Keep one row per key (latest by current_ts) to avoid multiple-match errors
        w = Window.partitionBy(*keys).orderBy(f.col("current_ts").desc())
        source_df = bronze_updt_df.withColumn("_rn", f.row_number().over(w)) \
                                  .filter("_rn = 1").drop("_rn")

        merge_condition = " AND ".join([f"t.`{k}` = s.`{k}`" for k in keys])

        (DeltaTable.forName(spark, target_table).alias("t")
            .merge(source_df.alias("s"), merge_condition)
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute())

else:
    raise ValueError(f"Unsupported load_type '{load_type}'. Use OVERWRITE, APPEND or MERGE.")

print(f"{load_type_norm or 'OVERWRITE'} load completed for {target_table}")

In [0]:
print(table_name)

In [0]:
%sql
select * from IDENTIFIER('man_catlog.caresync_schema_silver.' || :table_name) limit 10

In [0]:
record_count = bronze_updt_df.count()
print(f"Record count for {table_name}: {record_count}")
dbutils.notebook.exit(str(record_count))